##Read data##

In [17]:
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt

In [20]:
def process_file(file_name):
    data = loadmat(file_name)

    rr = data['rr'].flatten()
    target = data['targetsRR'].flatten()

    print(file_name)

    return rr, target
    

In [69]:
def evaluate(target, detect):
    TP = np.sum((target == 1) & (detect == 1))
    TN = np.sum((target == 0) & (detect == 0))
    FP = np.sum((target == 0) & (detect == 1))
    FN = np.sum((target == 1) & (detect == 0))

    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)

    return sensitivity, specificity


In [63]:
def detect_af(rr,target, window_size,threshold):
    print("Threshold used:", threshold)
    af_detected = []
    detected = 0
    rmssd_values = []
    for i in range(len(rr)-window_size+1):
        window = rr[i:i+window_size]
        diff = np.diff(window)
        rmssd = np.sqrt(np.mean(diff**2))
        rmssd_values.append(rmssd)
        if rmssd > threshold:
            af_detected.append(1)
        else:
            af_detected.append(0)   
    target_window = target[window_size-1:]
    af_detected = np.array(af_detected)
    return af_detected, target_window, rmssd_values



In [80]:

window_sizes = [5,10,15,20]
thresholds = [0.02, 0.03, 0.05, 0.08,0.10]
scores = []

best_window_size = None
best_threshold = None
best_score = 0
average_score = 0

for window_size in window_sizes:
    for threshold in thresholds:
        total_score = 0
        for i in range(1,5):
            rr, target = process_file(f'afdb_{i}.mat')
            af_detected,target_window, rmssd_values = detect_af(rr,target,window_size,threshold)
            sensitivity, specificity = evaluate(target_window,af_detected)
            score = (sensitivity + specificity)/2
            total_score += score
        average_score = total_score/4        
        print(
            "window_size =", window_size,
            "threshold =", threshold,
            "score =", average_score
        ) 
        if average_score > best_score:
            best_score = average_score
            best_threshold = threshold
            best_window_size = window_size
print("\nBest parameters:")
print("Window size:", best_window_size)
print("Threshold:", best_threshold)
print("Balanced accuracy:", best_score)
                
                




afdb_1.mat
Threshold used: 0.02
afdb_2.mat
Threshold used: 0.02
afdb_3.mat
Threshold used: 0.02
afdb_4.mat
Threshold used: 0.02
window_size = 5 threshold = 0.02 score = 0.758820988883463
afdb_1.mat
Threshold used: 0.03
afdb_2.mat
Threshold used: 0.03
afdb_3.mat
Threshold used: 0.03
afdb_4.mat
Threshold used: 0.03
window_size = 5 threshold = 0.03 score = 0.8067704462309762
afdb_1.mat
Threshold used: 0.05
afdb_2.mat
Threshold used: 0.05
afdb_3.mat
Threshold used: 0.05
afdb_4.mat
Threshold used: 0.05
window_size = 5 threshold = 0.05 score = 0.8439162720037074
afdb_1.mat
Threshold used: 0.08
afdb_2.mat
Threshold used: 0.08
afdb_3.mat
Threshold used: 0.08
afdb_4.mat
Threshold used: 0.08
window_size = 5 threshold = 0.08 score = 0.807650300950574
afdb_1.mat
Threshold used: 0.1
afdb_2.mat
Threshold used: 0.1
afdb_3.mat
Threshold used: 0.1
afdb_4.mat
Threshold used: 0.1
window_size = 5 threshold = 0.1 score = 0.7643628068955014
afdb_1.mat
Threshold used: 0.02
afdb_2.mat
Threshold used: 0.02
afd

In [81]:
fine_window_sizes = range(best_window_size - 2, best_window_size +2)
fine_thresholds = np.arange(best_threshold - 0.01, best_threshold + 0.02, 0.005)

fine_best_window_size = None
fine_best_threshold = None
fine_best_score = 0

for window_size in fine_window_sizes:
    for threshold in fine_thresholds:
        total_score = 0
        for i in range(1, 5):
            rr, target = process_file(f'afdb_{i}.mat')
            af_detected, target_window, _ = detect_af(
                rr, target, window_size, threshold
            )
            sensitivity, specificity = evaluate(
                target_window, af_detected
            )

            score = (sensitivity + specificity) / 2
            total_score += score  
        average_score = total_score / 4

        print(
            "fine window_size =", window_size,
            "fine threshold =", threshold,
            "score =", average_score
        )

        if average_score > fine_best_score:
            fine_best_score = average_score
            fine_best_threshold = threshold
            fine_best_window_size = window_size  
print("\nBest fine parameters:")
print("Window size:", fine_best_window_size)
print("Threshold:", fine_best_threshold)
print("Balanced accuracy:", fine_best_score)  

afdb_1.mat
Threshold used: 0.04
afdb_2.mat
Threshold used: 0.04
afdb_3.mat
Threshold used: 0.04
afdb_4.mat
Threshold used: 0.04
fine window_size = 8 fine threshold = 0.04 score = 0.8307426138788934
afdb_1.mat
Threshold used: 0.045
afdb_2.mat
Threshold used: 0.045
afdb_3.mat
Threshold used: 0.045
afdb_4.mat
Threshold used: 0.045
fine window_size = 8 fine threshold = 0.045 score = 0.8417426717280979
afdb_1.mat
Threshold used: 0.049999999999999996
afdb_2.mat
Threshold used: 0.049999999999999996
afdb_3.mat
Threshold used: 0.049999999999999996
afdb_4.mat
Threshold used: 0.049999999999999996
fine window_size = 8 fine threshold = 0.049999999999999996 score = 0.8486799622553161
afdb_1.mat
Threshold used: 0.05499999999999999
afdb_2.mat
Threshold used: 0.05499999999999999
afdb_3.mat
Threshold used: 0.05499999999999999
afdb_4.mat
Threshold used: 0.05499999999999999
fine window_size = 8 fine threshold = 0.05499999999999999 score = 0.8503236474968627
afdb_1.mat
Threshold used: 0.05999999999999999
a

In [82]:
print(sensitivity)
print(specificity)

0.8890938909389093
0.9420139623439814


In [84]:
for i in range(5, 8):   # test files 5,6,7
    rr, target = process_file(f'afdb_{i}.mat')

    af_detected, target_window, _ = detect_af(
        rr,
        target,
        fine_best_window_size,
        fine_best_threshold
    )

    sensitivity, specificity = evaluate(
        target_window,
        af_detected
    )

    balanced_accuracy = (sensitivity + specificity) / 2

    print(f'afdb_{i}.mat')
    print('Sensitivity:', sensitivity)
    print('Specificity:', specificity)
    print('Balanced accuracy:', balanced_accuracy)
    print()

afdb_5.mat
Threshold used: 0.05499999999999999
afdb_5.mat
Sensitivity: 0.9630996309963099
Specificity: 0.9350599544908342
Balanced accuracy: 0.9490797927435721

afdb_6.mat
Threshold used: 0.05499999999999999
afdb_6.mat
Sensitivity: 0.9723656240510173
Specificity: 0.4641168886978943
Balanced accuracy: 0.7182412563744558

afdb_7.mat
Threshold used: 0.05499999999999999
afdb_7.mat
Sensitivity: 0.9891167039160431
Specificity: 0.9164312617702448
Balanced accuracy: 0.952773982843144

